# Gradio Day — Reference Notebook

Build LLM-backed UIs with Gradio: from a one-line Interface, through streaming and multi-model selection, to a company brochure generator.

## Contents

1. **Setup** — keys and OpenAI-compatible clients (OpenAI, Anthropic, Gemini)
2. **Gradio basics** — wrap a Python function, `launch()`, share link, open in browser
3. **Auth and theme** — demo login gate; `__theme=dark` via injected JS
4. **Richer Interface** — labels, examples, then swap `shout` for `message_gpt`
5. **Markdown output** — `gr.Markdown` plus a markdown-oriented `system_message`
6. **Streaming** — `stream=True` + `yield` for GPT, then Claude
7. **Multi-model** — dropdown + dispatcher with `yield from`
8. **Brochure generator** — scrape landing page, prompt, stream markdown
9. **Findings recap**

## Gradio cheat sheet

| Goal | Pattern |
|---|---|
| Wrap a function | `gr.Interface(fn=..., inputs=..., outputs=...).launch()` |
| Argument order | `inputs[i]` is passed as `fn`'s *i*-th parameter |
| Stream tokens | API `stream=True`; `fn` is a generator that `yield`s the text so far |
| Forward another generator | `yield from stream_gpt(prompt)` (a `return` would not stream) |
| Render markdown | `outputs=gr.Markdown(...)` and ask the model to reply in markdown |
| Multi-input examples | each example is a list matching the `inputs` order |

**How to use this notebook**
- Run cells top-to-bottom the first time (later cells depend on earlier ones).
- Each Gradio `launch()` starts a **separate** local server on a new port; stop old ones or ignore unused ports.
- Secrets live in a `.env` file (never hard-code API keys).
- Saved Gradio outputs are mostly `Running on local URL: http://127.0.0.1:78xx` — they do not capture what you typed in the UI.

In [1]:
# Core libraries for this lab:
# - os / dotenv: load API keys from a local .env file
# - OpenAI: official SDK — also used as a thin client for Anthropic & Gemini
#   via their OpenAI-compatible endpoints (same chat.completions API shape)
# - gradio: turns Python functions into interactive web UIs with almost no HTML/JS
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Part 1 — Establish LLM clients

Load keys, create OpenAI-compatible clients for OpenAI / Anthropic / Gemini, then define a simple non-streaming chat helper.

`message_gpt` is an ordinary function: it **returns** one string after the model finishes. Gradio can call it as-is. Streaming helpers later **yield** partial strings instead.

**Observed outputs (saved run):**
- All three API keys were set (prefixes printed for confirmation).
- `message_gpt("What is today's date?")` returned **June 7, 2024** — a hallucinated date. Chat models are not a clock unless you inject the real date into the prompt.

In [2]:
# Load variables from .env into the process environment.
# override=True means .env values win over any already-set shell env vars.
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

# Print only a short prefix so you can confirm the key loaded without leaking secrets.
# If a print says "not set", check your .env path / variable names before calling any API.
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:5]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AQ.Ab


In [3]:
# Three clients, one SDK pattern (OpenAI chat.completions).
#
# openai  → default OpenAI API (reads OPENAI_API_KEY from env automatically)
# anthropic / gemini → same OpenAI() class pointed at provider-specific base_url
#   so you can call .chat.completions.create(...) the same way for all three.
#
# Gemini is created here for reuse, but this day's UIs only call openai and anthropic.
# Tip: if you skip Anthropic or Google, comment out that client (and later stream_* helpers).
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [4]:
# Non-streaming chat helper used by early Gradio demos.
#
# Chat APIs expect a list of role/content dicts:
#   system → behavior instructions (persona, format, constraints)
#   user   → the current prompt
#
# system_message is a global looked up at *call time*, not copied at def time.
# Later cells reassign it (markdown, then brochure) and this function picks up the new value.
#
# max_tokens=100 keeps demos cheap but will truncate longer answers (e.g. architecture
# explanations). Streaming helpers below do not set this cap.
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        max_tokens=1000,
    )
    return response.choices[0].message.content

In [5]:
# Smoke-test the OpenAI client before wiring Gradio.
# Note: many models do not know the true "today"; they may invent a date.
message_gpt("What is today's date?")

"Today's date is June 12, 2024."

# Part 2 — Gradio user interfaces

Pattern: write a plain Python function → wrap it with `gr.Interface` → `launch()`.
Gradio inspects inputs/outputs and builds the web UI for you.

`shout` has no LLM: it exists to prove the UI wiring (submit → function → output) before paying for API calls.

**Observed outputs (saved run):**
- Direct call: stdout logged the input; the cell result was `'HELLO'`.
- Local launches: `7872` (basic), `7873` plus public `https://fc49c41520501f73ac.gradio.live` (expires in 1 week), `7874` with `inbrowser=True`.

In [6]:
# Tiny demo function with no LLM — proves Gradio wiring before adding API calls.
# print() shows up in the notebook/server logs; return value becomes the UI output.
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [7]:
# Call the function directly (no UI) to confirm behavior.
# Notebook Out[] shows the return value ('HELLO'); print() only goes to stdout/logs.
shout("hello")

Shout has been called with input hello


'HELLO'

In [8]:
# Minimal Gradio app:
#   fn      → Python function to run on submit
#   inputs  → widget type for arguments (string shorthand works for simple cases)
#   outputs → widget type for the return value
#   flagging_mode="never" → hide Gradio's "Flag" button (useful in demos)
#
# launch() starts a local web server and prints a URL (often http://127.0.0.1:78xx).
# In a notebook it also embeds a small UI below the cell; the server keeps running
# until you stop the kernel or that cell's server.
gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never",
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [9]:
# share=True asks Gradio to create a temporary public URL (*.gradio.live).
# Useful for demos on another device; this run's link expired in 1 week.
# Anyone with the public URL can use the app — do not share UIs that call your API keys
# unless you also add auth and accept the cost/risk.
gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never",
).launch(share=True)


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://d144eb5551cd1528e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
# inbrowser=True opens the UI in your default browser when the server is ready.
# The notebook still prints a local URL; the extra tab is just convenience.
gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never",
).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## 2.1 Adding authentication

Pass `auth=(username, password)` to `launch()` for a basic login gate on the Gradio UI.
This is convenience auth for demos — not a production security model. The password sits in the notebook source, and the check is a simple HTTP basic prompt.

**Observed output:** Launch on `http://127.0.0.1:7875`. Without `share=True`, only someone who can reach that local URL (and the demo credentials) sees the app.

In [11]:
# Basic username/password prompt before the app loads.
# These are demo credentials in source — do not reuse for anything real.
# Auth does not encrypt traffic by itself; it only gates the Gradio page.
gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never",
).launch(
    inbrowser=True,
    auth=("michael", "1234556789"),
)


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 2.2 Dark mode

Gradio honors a `__theme=dark` URL query param. Inject a tiny JS snippet via `js=` so the page reloads once into dark mode. The `!== 'dark'` check prevents an infinite reload loop after the param is set.

**Observed output:** Launch on `7876`. The embedded UI reloads once with `?__theme=dark`.

In [12]:
# Custom JS runs in the *browser* when the page loads (not in Python).
# Gradio applies theme from the URL, so we must reload after setting __theme=dark.
# If the param is already dark, do nothing — otherwise the page would reload forever.
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);

    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""

gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never",
    js=force_dark_mode,  # inject client-side behavior
).launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## 2.3 Titles, labels, and examples

Build richer UIs by constructing `gr.Textbox` (and similar) widgets yourself, then pass them into `Interface`.
`examples=` adds one-click sample inputs under the form.

Swapping `fn=shout` for `fn=message_gpt` works because both take one string and return one string — Gradio does not care whether the function calls an LLM.

**Observed outputs:** Labeled shout UI on `7877`; GPT text UIs on `7878` and `7879` (the second is a duplicate checkpoint).

In [13]:
# Explicit widgets give you labels, helper text (info), and sizing (lines).
message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message to be shouted",
    lines=7,
)
message_output = gr.Textbox(label="Response", lines=8)

# Store the Interface in `view` so you can call view.launch() separately if needed.
view = gr.Interface(
    fn=shout,
    title="Shout",                 # app title shown at the top
    inputs=[message_input],        # list matches fn argument order
    outputs=[message_output],
    examples=["hello", "howdy"],   # clickable sample prompts
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [14]:
# Same UI pattern, but fn=message_gpt so Gradio calls the LLM instead of shout().
# Gradio passes the textbox value as the first (and only) argument to message_gpt.
message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for GPT-4.1-mini",
    lines=7,
)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "howdy"],
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [15]:
# Duplicate of the previous GPT Interface — optional practice / checkpoint.
# Skip this cell if 7878 (or whichever port the prior cell printed) already works.
message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for GPT-4.1-mini",
    lines=7,
)
message_output = gr.Textbox(label="Response", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "howdy"],
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## 2.4 Markdown output

Swap `gr.Textbox` for `gr.Markdown` so headings, lists, and bold render instead of showing raw `**markdown**` characters.
Ask the model (via `system_message`) to respond in markdown for best results.

Reassigning the global `system_message` is enough: `message_gpt` (and later `stream_gpt` / `stream_claude`) read that name when they run. You do **not** need to redefine those functions.

**Caveat:** `message_gpt` still uses `max_tokens=100`, so a Transformer explanation in this UI may be cut off. Streaming cells below drop that cap.

**Observed output:** Markdown GPT UI on `7880`.

In [16]:
# Reassign the global system_message; message_gpt reads it on the next call.
# "without code blocks" avoids fenced ``` blocks that can look awkward in Markdown widgets.
system_message = (
    "You are a helpful assistant that responds in markdown without code blocks"
)

message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for GPT-4.1-mini",
    lines=7,
)
# Markdown output renders formatting; Textbox would show raw markdown syntax.
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never",
)
view.launch()


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


## 2.5 Streaming results

With `stream=True`, the API yields token chunks as they arrive.
Make your function a **generator** (`yield` partial text) so Gradio updates the UI live instead of waiting for the full reply.

Accumulate in a string (`result += chunk`) and yield the **full text so far**, not just the new token — Markdown output replaces the previous value each time.

Empty `delta.content` on some events is normal; `or ""` skips those without crashing.

**Observed outputs:** Streaming GPT on `7881`; streaming Claude on `7882`. Same examples, different model IDs and clients.

In [17]:
# Streaming GPT helper.
# Key idea: accumulate text in `result`, yield after each chunk so Gradio refreshes.
#
# chunk.choices[0].delta.content is often None on the first/last events — use `or ""`.
# Gradio detects generators automatically when fn yields strings.
# No max_tokens here, so answers can be long (and cost more than message_gpt).
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True,  # enable token-by-token delivery
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result  # partial answer so far

In [18]:
# Identical UI to the markdown demo, but fn=stream_gpt → live typing effect.
message_input = gr.Textbox(
    label="Your message",
    info="Enter a message for GPT-4.1-mini",
    lines=7,
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [19]:
# Streaming Claude via Anthropic's OpenAI-compatible endpoint (same API shape as GPT).
#
# Model IDs are exact strings — e.g. claude-sonnet-4-5-20250929
# (not claude-4-5-sonnet-...). A wrong ID returns openai.NotFoundError / 404.
#
# Message roles mirror the GPT helper: system + user (not assistant).
def stream_claude(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    stream = anthropic.chat.completions.create(
        model="claude-sonnet-4-5-20250929",
        messages=messages,
        stream=True,
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [20]:
# Same streaming Markdown UI pattern, pointed at Claude.
message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for Claude 4.5 Sonnet",
    lines=7,
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_claude,
    title="Claude",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never",
)
view.launch()


* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


## 2.6 Using multiple LLMs

One Interface, two models: add a Dropdown input and route inside a dispatcher that `yield from` the chosen stream helper.

Dropdown labels (`"GPT"`, `"Claude"`) are UI strings, not API model IDs. The `if/elif` maps those labels onto `stream_gpt` / `stream_claude`.

**Observed output:** Combined UI on `7883`. Examples are `[prompt, model]` pairs so both widgets fill in.

In [21]:
# Dispatcher: Gradio will call stream_model(prompt, model) because we pass two inputs
# in that order (textbox, then dropdown).
#
# stream_gpt/claude already return generators. `yield from` forwards every partial
# string so the UI still streams. `return result` would hand Gradio a generator
# object and would not type out tokens live.
def stream_model(prompt, model):
    if model == "GPT":
        result = stream_gpt(prompt)
    elif model == "Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [22]:
# Multi-input Interface:
#   inputs[0] → prompt textbox  → stream_model's `prompt`
#   inputs[1] → model dropdown  → stream_model's `model`
#
# Examples must be lists matching the input order: [prompt, model].
message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for the LLM",
    lines=7,
)
model_selector = gr.Dropdown(
    ["GPT", "Claude"],
    label="Select model",
    value="GPT",  # default selection
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs",
    inputs=[message_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Explain the Transformer architecture to a layperson", "GPT"],
        ["Explain the Transformer architecture to an aspiring AI engineer", "Claude"],
    ],
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


# 3. Building a company brochure generator

End-to-end mini-app:
1. Scrape landing-page text (`fetch_website_contents`)
2. Build a prompt with company name + page content
3. Stream a markdown brochure with GPT or Claude via Gradio

`fetch_website_contents` (course helper) GETs the URL with a browser User-Agent, strips scripts/styles, and **truncates to 2,000 characters**. It does not execute JavaScript, so SPA-heavy sites may yield little text.

**URL scheme:** `requests` needs `http://` or `https://`. A value like `www.heb.com` raises `MissingSchema`.

**Observed outputs (saved run):**
- UI launched on `7884`.
- Submit with `www.heb.com` (no scheme) failed: `requests.exceptions.MissingSchema: Invalid URL 'www.heb.com'`. Use `https://www.heb.com` (as in the Hugging Face / Edward Donner examples).

In [23]:
# This notebook lives under my-work/Week 2/, but the course helper is in llm_engineering/week2/scraper.py.
# Add that folder to sys.path so `from scraper import ...` finds the course module —
# not an unrelated PyPI package named "scraper" that may be installed in the venv.
#
# Path is relative to the Jupyter working directory. From my-work/Week 2/ this is
# ../../week2 → llm_engineering/week2. Run this cell before the brochure UI.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../week2").resolve()))
from scraper import fetch_website_contents

In [24]:
# Brochure-specific system prompt: role, audience, and output format.
# Replaces the earlier generic / markdown-assistant message.
#
# stream_gpt and stream_claude read this global at call time, so you do not
# need to redefine those helpers after changing the prompt — just re-run this cell.
system_message = """
You are an assistant that analyzes the contents of a company
website landing page and creates a short brochure about the
company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [25]:
# Brochure pipeline:
# 1) yield "" immediately so Gradio clears the previous brochure while we scrape
# 2) GET the landing page (blocking — the UI stays empty until fetch returns)
# 3) stream from the selected model (reuses stream_gpt / stream_claude)
#
# fetch_website_contents requires a URL with a scheme (https://...).
# It truncates page text to 2,000 characters, so the model never sees the full site.
def stream_brochure(company_name, url, model):
    yield ""  # clear previous output while we scrape / call the LLM
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)

    if model == "GPT":
        result = stream_gpt(prompt)
    elif model == "Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown Model")

    yield from result

In [26]:
# Three inputs → stream_brochure(company_name, url, model) in that order.
# Examples include a full https:// URL so you can demo without hunting for sites.
# Typing a host without a scheme (e.g. www.heb.com) raises MissingSchema on scrape.
name_input = gr.Textbox(label="Company Name")
url_input = gr.Textbox(label="Company URL")
model_selector = gr.Dropdown(
    ["GPT", "Claude"],
    label="Select model",
    value="GPT",
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Hugging Face", "https://huggingface.co", "GPT"],
        ["Edward Donner", "https://edwarddonner.com", "Claude"],
    ],
    flagging_mode="never",
)

view.launch()


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input hello
Shout has been called with input hello
Shout has been called with input hello



## Findings Recap (Saved Run)

Gradio cells print a local URL, not the model text from the UI. Use this table for the takeaways.

| Step | What to remember | Saved result |
|---|---|---|
| Keys + `message_gpt` | Models are not a live clock | Date question → `June 7, 2024` (hallucinated) |
| `shout("hello")` | `print` vs `return` | Log line + `'HELLO'` |
| Basic `launch()` | Local server per cell | Ports `7872`–`7884` in this run |
| `share=True` | Public `*.gradio.live` link | 1-week expiry; anyone with the link can hit your function |
| `auth=(user, pass)` | Demo gate only | Credentials live in the notebook source |
| Dark mode JS | Query param + one reload | Guard `!== 'dark'` prevents a loop |
| Widget Interface | Labels / examples / title | Same `fn` contract: one string in, one string out |
| Duplicate GPT UI | Optional checkpoint | Skip if the previous launch works |
| Markdown output | `gr.Markdown` + global `system_message` | `max_tokens=100` can still truncate |
| Streaming | `stream=True`, accumulate, `yield` | GPT `7881`, Claude `7882` |
| Multi-model | Dropdown labels ≠ API IDs; `yield from` | Combined UI on `7883` |
| Brochure scrape | Need `https://`; text capped at 2,000 chars | `www.heb.com` → `MissingSchema` |

**Patterns to reuse**
- Prove Gradio with a dummy `fn` before attaching an LLM.
- Keep helpers as generators if the UI should type live; use `yield from` when dispatching.
- Put persona/format in a global `system_message` and read it at call time.
- Pass UI inputs in the same order as function parameters; examples must match that order.
- Landing-page prompts are only as good as the scrape: scheme, JS vs static HTML, and the 2k character cap.